# A-001 Simple RAG with LlamaIndex on Databricks

Goal: Answer questions from the 9 A-001 PDF documents stored in:

`/Workspace/Nuclear_Enterprise_360/A001 Documents`

Flow:

**PDFs → chunks → embeddings → vector index → top-3 retrieval → LLM → grounded answer**


In [0]:
%pip install -q -U \
    llama-index \
    llama-index-readers-file \
    llama-index-llms-databricks \
    llama-index-embeddings-databricks \
    pypdf

dbutils.library.restartPython()


Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
# ============================================================
# CONFIGURATION
# ============================================================

DOCUMENT_FOLDER = "/Workspace/Nuclear_Enterprise_360/A001 Documents"

CHAT_MODEL = "databricks-meta-llama-3-3-70b-instruct"
EMBED_MODEL = "databricks-qwen3-embedding-0-6b"

CHUNK_SIZE = 400
CHUNK_OVERLAP = 100
TOP_K = 3


In [0]:
# Check the files
import os

files = os.listdir(DOCUMENT_FOLDER)

print("Files found:")
for i, file in enumerate(files, 1):
    print(f"{i}. {file}")

print("\nTotal files:", len(files))


Files found:
1. 08_WO-2026-0817_A001_Inspection_Work_Order.pdf
2. 07_A001-CMR-2026-08_Condition_Monitoring_Report.pdf
3. 06_A001-TSG-001_Troubleshooting_and_Diagnostic_Guide.pdf
4. 04_A001-PROC-INS-001-V2_Approved_Inspection_and_PM_Procedure.pdf
5. 09_CR-2026-0819_A001_Field_Condition_Report.pdf
6. 02_A001-OPS-001_Operating_and_Usage_Guide.pdf
7. 01_REL-POL-001_Enterprise_Asset_Reliability_and_Escalation_Policy.pdf
8. 03_A001-PROC-INS-001-V1_Superseded_Inspection_Procedure.pdf
9. 05_A001-PROC-INS-001-V3D_Draft_Procedure.pdf

Total files: 9


In [0]:
# Configure Databricks LLM + embedding model
from llama_index.core import Settings
from llama_index.llms.databricks import Databricks
from llama_index.embeddings.databricks import DatabricksEmbedding

API_ROOT = (
    dbutils.notebook.entry_point
    .getDbutils()
    .notebook()
    .getContext()
    .apiUrl()
    .get()
)

API_TOKEN = (
    dbutils.notebook.entry_point
    .getDbutils()
    .notebook()
    .getContext()
    .apiToken()
    .get()
)

SERVING_ENDPOINT = f"{API_ROOT}/serving-endpoints"

llm = Databricks(
    model=CHAT_MODEL,
    api_key=API_TOKEN,
    api_base=SERVING_ENDPOINT,
)

embed_model = DatabricksEmbedding(
    model=EMBED_MODEL,
    api_key=API_TOKEN,
    endpoint=SERVING_ENDPOINT,
)

Settings.llm = llm
Settings.embed_model = embed_model

print("Models configured successfully.")


Models configured successfully.


In [0]:
# ============================================================
# CONFIGURE DATABRICKS LLM + EMBEDDING MODEL
# ============================================================
from llama_index.core import Settings
from llama_index.llms.databricks import Databricks
from llama_index.embeddings.databricks import DatabricksEmbedding

API_ROOT = (
    dbutils.notebook.entry_point
    .getDbutils()
    .notebook()
    .getContext()
    .apiUrl()
    .get()
)

API_TOKEN = (
    dbutils.notebook.entry_point
    .getDbutils()
    .notebook()
    .getContext()
    .apiToken()
    .get()
)

SERVING_ENDPOINT = f"{API_ROOT}/serving-endpoints"

# ============================================================
# CUSTOMIZE LLM SETTINGS
# ============================================================

# System prompt — defines the persona and rules for the LLM
SYSTEM_PROMPT = (
    "You are a knowledgeable assistant specialized in analyzing asset documentation. "
    "Answer questions based ONLY on the retrieved context from the provided documents. "
    "If the answer is not contained in the retrieved context, say \"I don't have enough "
    "information to answer this question based on the available documents.\" "
    "Do not hallucinate or make up information. Always cite the source file name "
    "when possible. Be concise, accurate, and professional."
)

# Generation parameters
TEMPERATURE = 0.1          # Lower = more focused & deterministic
MAX_TOKENS = 1024          # Maximum tokens in the generated response (max output tokens)

# Retrieval parameters
SIMILARITY_TOP_K = TOP_K   # Number of chunks to retrieve (inherits from config cell)

# ============================================================
# INITIALIZE MODELS WITH CUSTOM SETTINGS
# ============================================================

llm = Databricks(
    model=CHAT_MODEL,
    api_key=API_TOKEN,
    api_base=SERVING_ENDPOINT,
    temperature=TEMPERATURE,
    max_tokens=MAX_TOKENS,
)

embed_model = DatabricksEmbedding(
    model=EMBED_MODEL,
    api_key=API_TOKEN,
    endpoint=SERVING_ENDPOINT,
)

# Apply settings to LlamaIndex global Settings
Settings.llm = llm
Settings.embed_model = embed_model

# ============================================================
# PRINT CONFIGURATION SUMMARY
# ============================================================
print("Models configured successfully with custom settings:")
print(f"  Chat Model:     {CHAT_MODEL}")
print(f"  Embed Model:    {EMBED_MODEL}")
print(f"  Temperature:    {TEMPERATURE}")
print(f"  Max Tokens:     {MAX_TOKENS}")
print(f"  Top-K:          {SIMILARITY_TOP_K}")
print(f"  Chunk Size:     {CHUNK_SIZE}")
print(f"  Chunk Overlap:  {CHUNK_OVERLAP}")
print("\nSystem Prompt:")
print(SYSTEM_PROMPT)


In [0]:
# Load only PDF documents
from llama_index.core import SimpleDirectoryReader

documents = SimpleDirectoryReader(
    input_dir=DOCUMENT_FOLDER,
    recursive=True,
    required_exts=[".pdf"]
).load_data()

print("Documents loaded:", len(documents))


Documents loaded: 17


In [0]:
# Chunk the documents
from llama_index.core.node_parser import SentenceSplitter

splitter = SentenceSplitter(
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP
)

Settings.text_splitter = splitter

print("Chunk size:", CHUNK_SIZE)
print("Chunk overlap:", CHUNK_OVERLAP)


Chunk size: 400
Chunk overlap: 100


In [0]:
# Build the vector index
from llama_index.core import VectorStoreIndex

index = VectorStoreIndex.from_documents(
    documents,
    transformations=[splitter],
    show_progress=True
)

print("RAG index created.")


Applying transformations:   0%|          | 0/1 [00:00<?, ?it/s]

Generating embeddings:   0%|          | 0/27 [00:00<?, ?it/s]

2026-09-22 07:24:06,716 - INFO - HTTP Request: POST https://dbc-45ef533b-b71f.cloud.databricks.com/serving-endpoints/embeddings "HTTP/1.1 200 OK"


RAG index created.


In [0]:
# Create query engine
query_engine = index.as_query_engine(
    similarity_top_k=TOP_K
)

print("Query engine ready.")


Query engine ready.


In [0]:
# Ask a first question
question = "What is asset A-001?"

response = query_engine.query(question)

print("QUESTION:")
print(question)

print("\nANSWER:")
print(response)


2026-09-22 06:30:47,808 - INFO - HTTP Request: POST https://dbc-45ef533b-b71f.cloud.databricks.com/serving-endpoints/embeddings "HTTP/1.1 200 OK"
2026-09-22 06:30:48,585 - INFO - HTTP Request: POST https://dbc-45ef533b-b71f.cloud.databricks.com/serving-endpoints/chat/completions "HTTP/1.1 200 OK"


QUESTION:
What is asset A-001?

ANSWER:
Asset A-001 is a Cooling Water Pump.


In [0]:
# Try another question
question = "what is the location of a-001?"

response = query_engine.query(question)

print(response)


2026-09-22 07:24:14,620 - INFO - HTTP Request: POST https://dbc-45ef533b-b71f.cloud.databricks.com/serving-endpoints/embeddings "HTTP/1.1 200 OK"
2026-09-22 07:24:15,192 - INFO - HTTP Request: POST https://dbc-45ef533b-b71f.cloud.databricks.com/serving-endpoints/chat/completions "HTTP/1.1 200 OK"


The location of A-001 is the Utilities Area - Bay 3.


In [0]:
# Show retrieved sources
question = "who are the stakeholders here any name or contact"

response = query_engine.query(question)

print("=" * 80)
print("ANSWER")
print("=" * 80)
print(response)

print("\n" + "=" * 80)
print("SOURCES RETRIEVED")
print("=" * 80)

for i, node in enumerate(response.source_nodes, 1):
    file_name = node.node.metadata.get("file_name", "Unknown file")

    print(f"\nSOURCE {i}")
    print("File:", file_name)
    print("Similarity score:", round(node.score, 4) if node.score else "N/A")

    print("\nRetrieved text:")
    print(node.node.text[:700])

    print("-" * 80)


2026-09-22 07:27:29,525 - INFO - HTTP Request: POST https://dbc-45ef533b-b71f.cloud.databricks.com/serving-endpoints/embeddings "HTTP/1.1 200 OK"
2026-09-22 07:27:32,777 - INFO - HTTP Request: POST https://dbc-45ef533b-b71f.cloud.databricks.com/serving-endpoints/chat/completions "HTTP/1.1 200 OK"


ANSWER
There are no specific names or contact information mentioned for the stakeholders. However, the stakeholders can be identified based on their roles, which include:

1. Maintenance Planning - The owner of Asset A-001.
2. Condition Monitoring Team - The team that requested the work order.
3. Operations Support - The owner of the Field Condition Report.
4. Operations Technician - The observer who recorded the intermittent rattling sound and small wet area near the baseplate.
5. Reviewer - The person who will review and approve the draft procedure.
6. Asset Reliability Training Scenario team - The team responsible for creating the synthetic training documents.

SOURCES RETRIEVED

SOURCE 1
File: 08_WO-2026-0817_A001_Inspection_Work_Order.pdf
Similarity score: 0.3747

Retrieved text:
NORTHSTAR PROCESS FACILITY SYNTHETIC TRAINING CORPUS
WO-2026-0817  |  OPEN  |  Synthetic training document - not for operational use
Page 1
WO-2026-0817  /  OPEN
Inspection Work Order - A-001 Elevated 
Vi

In [0]:
# ============================================================
# CUSTOMIZE LLAMAINDEX SETTINGS
# ============================================================
# System prompt — defines the persona and rules for the LLM
SYSTEM_PROMPT = (
    "You are a knowledgeable assistant specialized in analyzing asset documentation. "
    "Answer questions based ONLY on the retrieved context from the provided documents. "
    "If the answer is not contained in the retrieved context, say \"I don't have enough "
    "information to answer this question based on the available documents.\" "
    "Do not hallucinate or make up information. Always cite the source file name "
    "when possible. Be concise, accurate, and professional."
)

# Generation parameters
TEMPERATURE = 0.1          # Lower = more focused & deterministic
MAX_TOKENS = 1024          # Maximum tokens in the generated response

# Retrieval parameters
SIMILARITY_TOP_K = 3       # Number of chunks to retrieve

# ============================================================
# APPLY SETTINGS
# ============================================================
from llama_index.core import Settings

# Apply generation settings to the LLM
llm_with_settings = Databricks(
    model=CHAT_MODEL,
    api_key=API_TOKEN,
    api_base=SERVING_ENDPOINT,
    temperature=TEMPERATURE,
    max_tokens=MAX_TOKENS,
)

Settings.llm = llm_with_settings

print("LLM settings applied:")
print(f"  Model:        {CHAT_MODEL}")
print(f"  Temperature:  {TEMPERATURE}")
print(f"  Max tokens:   {MAX_TOKENS}")
print(f"  Top-K:        {SIMILARITY_TOP_K}")

# ============================================================
# REBUILD QUERY ENGINE WITH CUSTOM SETTINGS
# ============================================================
from llama_index.core.prompts import ChatPromptTemplate
from llama_index.core.llms import ChatMessage, MessageRole

# Build a chat prompt template that includes the system prompt
text_qa_template = ChatPromptTemplate(
    message_templates=[
        ChatMessage(role=MessageRole.SYSTEM, content=SYSTEM_PROMPT),
        ChatMessage(
            role=MessageRole.USER,
            content=(
                "Context information is below.\n"
                "---------------------\n"
                "{context_str}\n"
                "---------------------\n"
                "Given the context information and not prior knowledge, "
                "answer the query.\n"
                "Query: {query_str}\n"
                "Answer: "
            ),
        ),
    ]
)

query_engine = index.as_query_engine(
    similarity_top_k=SIMILARITY_TOP_K,
    text_qa_template=text_qa_template,
)

print("\nQuery engine rebuilt with custom system prompt and settings.")
print("System prompt:\n")
print(SYSTEM_PROMPT)

LLM settings applied:
  Model:        databricks-meta-llama-3-3-70b-instruct
  Temperature:  0.1
  Max tokens:   1024
  Top-K:        3

Query engine rebuilt with custom system prompt and settings.
System prompt:

You are a knowledgeable assistant specialized in analyzing asset documentation. Answer questions based ONLY on the retrieved context from the provided documents. If the answer is not contained in the retrieved context, say "I don't have enough information to answer this question based on the available documents." Do not hallucinate or make up information. Always cite the source file name when possible. Be concise, accurate, and professional.
